Finding number of connections for superheros 

In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

spark = SparkSession.builder.appName('Superheros').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

namesDF = spark.read.schema(schema).option('sep', ' ').csv('../MarvelNames.txt')
linesDF = spark.read.text('../MarvelGraph.txt')

connections = linesDF.withColumn('id', F.split(F.col('value'), ' ')[0]) \
.withColumn('connections', F.size(F.split(F.col('value'), ' ')) - 1) \
.groupBy('id').agg(F.sum('connections').alias('connections'))

mostPopular = connections.sort(F.col('connections').desc())

connections.show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 12:02:47 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 12:02:47 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 12:02:48 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


+----+-----------+
|  id|connections|
+----+-----------+
| 691|          7|
|1159|         12|
|3959|        143|
|1572|         36|
|2294|         15|
|1090|          5|
|3606|        172|
|3414|          8|
| 296|         18|
|4821|         17|
|2162|         42|
|1436|         10|
|1512|         12|
|6194|         15|
|6240|         12|
| 829|         38|
|2136|          7|
|5645|         21|
|2069|        264|
| 467|          1|
+----+-----------+
only showing top 20 rows


Finding most popular hero. Adding in graph to connect names to superheros


In [ ]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

spark = SparkSession.builder.appName('Superheros').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

namesDF = spark.read.schema(schema).option('sep', ' ').csv('../MarvelNames.txt')
linesDF = spark.read.text('../MarvelGraph.txt')

connections = linesDF.withColumn('id', F.split(F.col('value'), ' ')[0]) \
.withColumn('connections', F.size(F.split(F.col('value'), ' ')) - 1) \
.groupBy('id').agg(F.sum('connections').alias('connections'))

mostPopular = connections.sort(F.col('connections').desc()).first()
mostPopularName = namesDF.filter(F.col('id') == mostPopular[0]).select('name').first()

print(mostPopularName[0] + ' is the most popular hero, with ' + str(mostPopular[1]) + ' appearances!')

# connections.show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 11:51:21 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 11:51:21 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 11:51:22 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


CAPTAIN AMERICA is the most popular hero, with 1937 appearances!


Challenge: Find all obscure heros, keeping in mind that there may be ties
Hints: 
Dataframename.agg(func.min(‘columnName’).first()[0]

Dataframename.filter(func.col(‘columnName’) == someValue)

Dataframename.join(otherDataframeWithACommonColumnName, ‘commonColumnName’)


In [1]:
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructField, StructType, IntegerType, StringType

spark = SparkSession.builder.appName('Superheros').getOrCreate()

schema = StructType([
    StructField('id', IntegerType()),
    StructField('name', StringType())
])

namesDF = spark.read.schema(schema).option('sep', ' ').csv('../MarvelNames.txt')
linesDF = spark.read.text('../MarvelGraph.txt')

connections = linesDF.withColumn('id', F.split(F.col('value'), ' ')[0]) \
.withColumn('connections', F.size(F.split(F.col('value'), ' ')) - 1) \
.groupBy('id').agg(F.sum('connections').alias('connections'))

# Step 1: find the minimum connection count as a single value
minConnections = connections.agg(F.min('connections')).first()[0]

# Step 2: filter for every hero whose count equals that minimum
leastPopular = connections.filter(F.col('connections') == minConnections)

# Step 3: join with names to get readable output
leastPopularNames = leastPopular.join(namesDF, on='id')

print(f"Hero(es) with the fewest connections ({minConnections}):")
leastPopularNames.select('name', 'connections').show(truncate=False)



# mostPopular = connections.sort(F.col('connections').desc()).first()
# mostPopularName = namesDF.filter(F.col('id') == mostPopular[0]).select('name').first()

# print(mostPopularName[0] + ' is the most popular hero, with ' + str(mostPopular[1]) + ' appearances!')

# connections.show()

spark.stop()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/07/06 13:16:10 WARN Utils: Your hostname, CartersDell, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/07/06 13:16:10 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/07/06 13:16:10 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Hero(es) with the fewest connections (1):
+--------------------+-----------+
|name                |connections|
+--------------------+-----------+
|BERSERKER II        |1          |
|BLARE/              |1          |
|MARVEL BOY II/MARTIN|1          |
|MARVEL BOY/MARTIN BU|1          |
|GIURESCU, RADU      |1          |
|CLUMSY FOULUP       |1          |
|FENRIS              |1          |
|RANDAK              |1          |
|SHARKSKIN           |1          |
|CALLAHAN, DANNY     |1          |
|DEATHCHARGE         |1          |
|RUNE                |1          |
|SEA LEOPARD         |1          |
|RED WOLF II         |1          |
|ZANTOR              |1          |
|JOHNSON, LYNDON BAIN|1          |
|LUNATIK II          |1          |
|KULL                |1          |
|GERVASE, LADY ALYSSA|1          |
+--------------------+-----------+

